# 2. Investigating Momentum

### Data

In this section, we empirically investigate some concerns regarding AQR's new momentum product.
On Canvas, find the data file, `data/momentum_data.xlsx`.

* The first tab contains the momentum factor as an excess return: $\tilde{r}^{\mathrm{mom}}$.
* The second tab contains returns on portfolios corresponding to scored momentum deciles.
  - $r^{\operatorname{mom}(1)}$ denotes the portfolio of stocks in the lowest momentum decile, the "losers" with the lowest past returns.
  - $r^{\operatorname{mom}(10)}$ denotes the portfolio of stocks in the highest momentum decile.
* The third tab gives portfolios sorted by momentum and size.
  - $r^{\text {momSU }}$ denotes the portfolio of small stocks in the top 3 deciles of momentum scores.
  - $r^{\text {momBD }}$ denotes the portfolio of big-stocks in the bottom 3 deciles of momentum scores.

Note that the Fama-French momentum return, $\tilde{r}^{\mathrm{mom}: \mathrm{FF}}$, given in the first tab, is constructed by $\mathrm{FF}$ as,

$$
\tilde{r}^{\mathrm{mom}: \mathrm{FF}}=\frac{1}{2}\left(r^{\mathrm{momBU}}+r^{\mathrm{momSU}}\right)-\frac{1}{2}\left(r^{\mathrm{momBD}}+r^{\mathrm{momSD}}\right)
$$

### 2.1 Is Momentum still profitable?

The excess returns of lucrative trading strategies often disappear once the strategy is well-known. The first widely-cited paper on momentum was published in 1993. Have momentum returns or risk changed since then? The AQR case takes place at the end of 2008. Have momentum returns changed in 2009-2024?

Investigate by filling out the summary statistics below for the full-sample and three sub-samples.

#### (a) 
Using the data provided, fill in Table 1 with the appropriate stats for $\tilde{r}^{\text {mom:FF }}$.

$$
\begin{equation*}
\begin{aligned}
&\text { Table 1: Long-short momentum performance. }\\
&\begin{array}{lllllll}
\hline \hline \text { Subsample } & \text { mean } & \text { vol } & \text { Sharpe } & \text { skewness } & \text { corr. to } \tilde{r}^m & \text { corr. to } \tilde{r}^v \\
\hline \hline
1927-2024 & & & & & & \\
1927-1993 & & & & & & \\
1994-2008 & & & & & & \\
2009-2024 & & & & & & \\
\hline \hline
\end{array}
\end{aligned}
\end{equation*}
$$

In [4]:
import pandas as pd 
import numpy as np 

DATA = "data/momentum_data.xlsx"
factors = pd.read_excel(DATA, sheet_name = 'factors (excess returns)', index_col = 'Date', parse_dates = True)
momentum = pd.read_excel(DATA, sheet_name = 'momentum (excess returns)', index_col = 'Date', parse_dates = True)
deciles = pd.read_excel(DATA, sheet_name = 'deciles (total returns)', index_col = 'Date', parse_dates = True)
size_sorts = pd.read_excel(DATA, sheet_name = 'size_sorts (total returns)', index_col = 'Date', parse_dates = True)
risk_free = pd.read_excel(DATA, sheet_name = 'risk-free rate', index_col = 'Date', parse_dates = True)




In [5]:
df = factors.join(momentum)   # MKT, SMB, HML, UMD in one table

#a dictionary: a list of labels, each paired with a value (start and end year)
subsamples = {
    '1927-2024': ('1927', '2024'),
    '1927-1993': ('1927', '1993'),
    '1994-2008': ('1994', '2008'),
    '2009-2024': ('2009', '2024'),
}

#r: the strategy's monthly returns (UMD)
#mkt: the market's monthly returns (MKT)
#val: the value factor's monthly returns (HML)
def summary_stats(r, mkt, val):
    return {
        'mean':      r.mean() * 12,
        'vol':       r.std() * np.sqrt(12),
        'Sharpe':    (r.mean() / r.std()) * np.sqrt(12),
        'skewness':  r.skew(),
        'corr. to MKT': r.corr(mkt),
        'corr. to HML': r.corr(val),
    }

table1 = {}
for name, (start, end) in subsamples.items():
    d = df.loc[start:end] #slice out just the years you want to look at per iteration
    table1[name] = summary_stats(d['UMD'], d['MKT'], d['HML'])

table1 = pd.DataFrame(table1).T
table1.round(4)


,mean,vol,Sharpe,skewness,corr. to MKT,corr. to HML
1927-2024,0.0739,0.1627,0.4540,-3.0673,-0.3492,-0.4098
1927-1993,0.0881,0.1605,0.5491,-3.9110,-0.3675,-0.5010
1994-2008,0.1027,0.1754,0.5858,-0.5964,-0.2402,-0.1198
2009-2024,-0.0129,0.1580,-0.0816,-2.6916,-0.3702,-0.3190


#### (b)
Has momentum changed much over time, as seen through these subsample statistics?


It appears that momentum did not disappear after it became well known in 1993. From 1994-2008, it had its highest mean of about 10.3% and sharpe ratio of 0.59. However, the mean return has shifted since the time of the case: from 2009-2024, the mean excess return and sharpe were negative, and close to zero. Momentum earned little to nothing in that period. 

Volatility has stayed relatively stable in every subsample, and skewness was strongly negative in most periods as well. This means that momentum is prone to rare but very large crashes. 

Correlations with the market and with value were negative in every subsample. They were weaker in 1994-2008, but stronger again after 2009. Overall, the risk and diversification properties of momentum have been conmsistent over time, but its average return has dropped significantly since 2009.

#### (c) 

Does this data support AQR's argument that momentum is an important piece of the ideal portfolio? What if mean returns to momentum are in actuality near zero due to transaction costs - would there still be evidence here that momentum is valuable?

Over the sample, the data does support AQR's argument. Over the whole sample, momentum earned a 7.4% annual excess return with a sharpe ratio of 0.45, comparable to the market itself. Additionally, it was negatively correlated with both the market and value. So, even if transaction costs brought momentum's mean return close to zero, there is still evidence for momentum's value. The negative correlations suggest an argument that it could function for diversification and reduce the overall risk of the portfolio. The main concern might be momentum's large negative skewness. Namely, momentum has rare but severe crashes. Combined with a near-zero mean return, this means investors should not rely too heavily on momentum because it poses a potentially large risk with small consistent returns. 

### 2.2. Whether a long-only implementation of momentum is valuable.

Construct your own long-only implementation:

$$\tilde{r}^{momU:FF} = \frac{1}{2} (\tilde{r}^{momBU} + \tilde{r}^{momSU}) − r^f $$

Note that this is following the FF approach of treating big and small stocks separately. This would be very similar to a scaled version of,

$$\tilde{r}^{momU} = \frac{1}{3} (\tilde{r}^{mom(8)} + \tilde{r}^{mom(9)} + \tilde{r}^{mom(10)}) − r^f $$

For the question below, use the FF-style $\tilde{r}^{momU:FF}_t$

#### (a) 

Fill out Table 2 for the data in the period 1994-2024.

$$
\begin{equation*}
\begin{aligned}
&\text { Table 2: Long-only momentum performance. }\\
&\begin{array}{lllllll}
\hline \hline \text { 1994-2024 } & \text { mean } & \text { vol } & \text { Sharpe } & \text { skew } & \text { corr. to } \tilde{r}^m & \text { corr. to } \tilde{r}^v \\
\hline \hline \text { Long-and-short, }\left(\tilde{r}^{\text {mom:FF }}\right) & & & & & & \\
\text { Long-only }\left(\tilde{r}^{\text {momU:FF }}\right) & & & & & & \\
\hline \hline
\end{array}
\end{aligned}
\end{equation*}
$$

#### (b)  
Is long-only momentum as attractive as long-short momentum with respect to mean, volatility, and Sharpe Ratio?

#### (c) 
Is long-only momentum as diversifying as long-short momentum with respect to market and value premia?

#### (d) 
Show a plot of the cumulative product of $1+\tilde{r}^{\text {mom:FF }}$ and $1+\tilde{r}^{\text {momU:FF }}$ over the 1994-2024 subsample.

### 2.3. 
Is momentum just data mining, or is it a robust strategy?

Assess how sensitive the threshold for the “winners” and “losers” is in the results. Specifically, we compare three constructions:

- long the top 1 decile and short the bottom 1 deciles:

$$ \tilde{r}^{momD1} = r^{mom(10)} - r^{mom(1)}$$

- long the top 3 deciles and short the bottom 3 deciles:

$$ 
\begin{aligned}
\tilde{r}^{momD3} &= \frac{1}{3}(r^{mom(8)} + r^{mom(9)} + r^{mom(10)}) - \frac{1}{3} (r^{mom(3)} + r^{mom(2)} + r^{mom(1)})\\
&= \frac{1}{3}\sum_{k = 8}^{10} r^{mom(k)} - \frac{1}{3}\sum_{k = 1}^{3} r^{mom(k)} 
\end{aligned}$$

- long the top 5 deciles and short the bottom 5 decile:

$$ \tilde{r}^{momD5} = \frac{1}{5}\sum_{k = 6}^{10} r^{mom(k)} - \frac{1}{5}\sum_{k = 1}^{5} r^{mom(k)} $$

#### (a) 
Compare all three constructions by filling out the stats in the table below for the period 1994-2024.

$$
\begin{equation}
\begin{aligned}
&\text { Table 3: Robustness of momentum construction. }\\
&\begin{array}{lllllll}
\hline \hline \text { 1994-2024 } & \text { mean } & \text { vol } & \text { Sharpe } & \text { skewness } & \text { corr. to } \tilde{r}^m & \text { corr. to } \tilde{r}^v \\
\hline \hline \tilde{r}^{\text {momD1 }} & & & & & & \\
\tilde{r}^{\text {momD3 }} & & & & & & \\
\tilde{r}^{\text {momD5 }} & & & & & & \\
\hline \hline
\end{array}
\end{aligned}
\end{equation}
$$

#### (b) 
Do the tradeoffs between the 1-decile, 3-decile, and 5-decile constructions line up with the theoretical tradeoffs we discussed in the lecture?

#### (c) 
Should AQR's retail product consider using a 1-decile or 5-decile construction?

#### (d) 
Does $\tilde{r}^{\text {momD3 }}$ have similar stats to the Fama-French construction in (1). Recall that construction is also a 3-decile, long-short construction, but it is segmented for small and large stocks. Compare the middle row of Table 3 with the top row of Table 2.

### 2.4. 
Does implementing momentum require trading lots of small stocks-- thus causing even larger trading costs?

For regulatory and liquidity reasons, AQR is particularly interested in using larger stocks for their momentum baskets. (Though they will launch one product that focuses on medium-sized stocks.)

Use the data provided on both small-stock “winners”, $r^{momSU}$, and small-stock “losers”, $r^{momSD}$, to construct a small-stock momentum portfolio,

$$ r^{momS}_t = r^{momSU}_t - r^{momSD}_t $$

Similarly, use the data provided to construct a big-stock momentum portfolio,

$$ r^{momB}_t = r^{momBU}_t − r^{momBD}_t $$

#### (a) 
Fill out Table 4 over the sample 1994-2024.

$$
\begin{equation*}
\begin{aligned}
&\text { Table 4: Momentum of small and big stocks. }\\
&\begin{array}{llllll}
\hline \hline \text { 1994-2024 } & \text { mean } & \text { vol } & \text { Sharpe } & \text { skewness } & \text { corr. to } \tilde{r}^m \\
\hline \hline \text { All stocks, } \tilde{r}^{\text {mom:FF }} & & & & & \\
\text { Small stocks } r_t^{\text {mom } S} & & & & & \\
\text { Large stocks } r_t^{\text {mom } B} & & & & & \\
\hline
\end{array}
\end{aligned}
\end{equation*}
$$

#### (b) 
Is the attractiveness of the momentum strategy mostly driven by the small stocks? That is, does a momentum strategy in large stocks still deliver excess returns at comparable risk?

### 2.5. 
In conclusion, what is your assessment of the AQR retail product? Is it capturing the important features of the Fama-French construction of momentum? Would you suggest any modifications?